In [1]:
R.<q,t,a> = LaurentPolynomialRing(QQ,3)
def Absolut(U):    #counts number of 1's in U
    A=0
    for x in U:
        if x==1:
            A=A+1
    return A
def HHH(U,V):      #recursion for HHH 
#    print(U)
#    print(V)
    if Absolut(U)==0:
        U1=U[1:]
        U1.append(1)
        V1=V[1:]
        V1.append(1)
        return (1-q)^(-1)*HHH(U1,V1)
    if U[0]==0 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        V1=V[1:]
        V1.append(1)
        U0=U[1:]
        U0.append(0)
        V0=V[1:]
        V0.append(0)
        return t^(-Absolut(U))*(HHH(U1,V1)+q*HHH(U0,V0))
    if U[0]==1 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        return HHH(U1,V[1:])
    if U[0]==0 and V[0]==1:
        V1=V[1:]
        V1.append(1)
        return HHH(U[1:],V1)
    if U[0]==1 and V[0]==1:
        if len(U)>1 or len(V)>1:
            return (t^(Absolut(U)-1)+a)*HHH(U[1:],V[1:])
        else:
            return 1+a

def IsAdm(A,n,d):  # A is a subset of Z/ndZ; this tests if A is admissible: it is inadmissible if it contains all elements in a congruence class mod d.
    test=0
    for i in range(d):
        test=0
        for j in range(n):
            if d*j+i in A:
                test=test+1
            if test==n:
                return False
    return True
def Aminus(A,n,d): # A is a subset of Z/ndZ; this returns A-1 (mod nd)
    B=[]
    for i in A:
        if i==0:
            B.append(n*d-1)
        else:
            B.append(i-1)
    return B
def CJPoinc(U,V,A,n,m,d):  # recursion computing the (nd,md) q,t-Catalan polynomial via admissible nd,md - invariant subsets
    if not IsAdm(A,n,d):
        return 0
    test=True
    for i in U:
        if not i==2:
            test=False
    for i in V:
        if not i==2:
            test=False
    if test:
        return 1
    if U[0]==0 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        V1=V[1:]
        V1.append(1)
        U0=U[1:]
        U0.append(0)
        V0=V[1:]
        V0.append(0)
        if U[len(U)-1]==2:
            B=Aminus(A,n,d)
            if not 0 in B:
                B.append(0)
            return t^(-Absolut(U))*(CJPoinc(U1,V1,Aminus(A,n,d),n,m,d)+q*CJPoinc(U0,V0,B,n,m,d))
        else:
            return t^(-Absolut(U))*(CJPoinc(U1,V1,Aminus(A,n,d),n,m,d)+q*CJPoinc(U0,V0,Aminus(A,n,d),n,m,d))
    if U[0]==1 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        V2=V[1:]
        V2.append(2)
        return CJPoinc(U1,V2,Aminus(A,n,d),n,m,d)
    if U[0]==0 and V[0]==1:
        U2=U[1:]
        U2.append(2)
        V1=V[1:]
        V1.append(1)
        return CJPoinc(U2,V1,Aminus(A,n,d),n,m,d)
    if U[0]==1 and V[0]==1:
        U2=U[1:]
        U2.append(2)
        V2=V[1:]
        V2.append(2)
        return (t^(Absolut(U)-1)+a)*CJPoinc(U2,V2,Aminus(A,n,d),n,m,d)
    if U[0]==2 and V[0]==2:
        U2=U[1:]
        U2.append(2)
        V2=V[1:]
        V2.append(2)
        return CJPoinc(U2,V2,Aminus(A,n,d),n,m,d)
def PMax(n,m,d):  #returns maximal (or is it minimal?) nd,md Dyck path
    P=[]
    for i in range(n*d):
        P.append(floor(m*d-(i+1)*m*n^(-1)))
    return Partition(P)
def Ngens(P,n,m,d): # returns the set of rightmost boxes inside in rows of P (here P is treated as a lattice path crossing nd x Z strip, so this returns nd boxes)
    A=[]
    for i in range(n*d):
        if i<P.length():
            A.append((i,P[i]-1))
        else:
            A.append((i,-1))
    return A
def Mcogens(P,n,m,d): # return the set of the lowest boxes outside of P (here P is treated as a lattice path inside a Z x md strip, so this returns md boxes)
    A=[]
    if P.length()>0:
        for j in range(P[0],m*d):
            A.append((0,j))
        for i in range(P.length()-1):
            for j in range(P[i+1],P[i]):
                A.append((i+1,j))
        for j in range(P[P.length()-1]):
            A.append((P.length(),j))
    else:
        for j in range(m*d):
            A.append((0,j))
    return A
def Rank(A,n,m,d): # A is a box; the linear rank function normilized so that Rank(-1,md-1)=Rank(nd-1,-1)=0
    return n*m*d-n-m-A[0]*m-A[1]*n
def IsAbB(A,B,n,m,d): # A and B are boxes; answers to 'Is A > B according to rank with the first coordinate as tiebreaker' 
    if Rank(A,n,m,d)>Rank(B,n,m,d):
        return True
    elif Rank(A,n,m,d)<Rank(B,n,m,d):
        return False
    elif A[0]<B[0]:
        return True
    else:
        return False
def codinv(P,n,m,d): # Compute codinv(P) using the order on generators and cogenerators
    x=0
    A=Ngens(P,n,m,d)
    B=Mcogens(P,n,m,d)
    for a in A:
        for b in B:
            if IsAbB(b,a,n,m,d):
                x=x+1
    return x
def dinv(P,n,m,d): # Computes dinv using Haiman's leg-arm formula
    x=0
    for c in P.cells():
        if P.leg_length(c[0],c[1])*m < (P.arm_length(c[0],c[1])+1)*n and (P.leg_length(c[0],c[1])+1)*m >= P.arm_length(c[0],c[1])*n:
            x=x+1
    return x
def afact(P,n,m,d): # computes the factor for getting the higher a-power; Fixed!
    dcogens=P.addable_cells()
    B=Mcogens(P,n,m,d)
    f=1
    for g in dcogens:
        x=0
        for b in B:
            if IsAbB(b,g,n,m,d) and IsAbB((g[0],g[1]-1),b,n,m,d):
                x=x+1
        f=f*(1+a*t^(-x))
    return f
def qtCat(n,m,d): # computes the q,t-Catalan (with a=0); with general a computes the q,t - Schroeder polynomial 
    Max=PMax(n,m,d)
    delt=Max.size()
    f=0
    for s in range(delt+1):
        for P in Partitions(s,outer=Max):
            f=f+q^(delt-s)*t^dinv(P,n,m,d)*afact(P,n,m,d)
    return f
def CJinput(n,m,d): # generates (0001,000001) input for the СJPoinc recursion 
    A=[]
    B=[]
    for i in range(n*d-1):
        A.append(0)
    A.append(1)
    for i in range(m*d-1):
        B.append(0)
    B.append(1)
    return [A,B]
def HHHinput(n,m,d): # generates (0100,0000001) input for the HHH recursion
    A=[]
    B=[]
    for i in range(d-1):
        A.append(0)
    A.append(1)
    for i in range((n-1)*d):
        A.append(0)
    for i in range(m*d):
        B.append(0)
    B.append(1)
    return [A,B]
R=QQ['q','t','a']
q=R.0
t=R.1
a=R.2
# Definitions of coarm and coleg
def coarm(tab,i):
    return tab.cells_containing(i)[0][0]
def coleg(tab,i):
    return tab.cells_containing(i)[0][1]
# (q,t)-contents of boxes of a skew tableau
def zz(tab):
    size=tab.size()
    return [q^coarm(tab,i)*t^coleg(tab,i) for i in range(1,size+1)]
# definition of omega
def std(x):
    if x==0: 
        return 1
    else:
        return x
def om(x):
    return std(1-x)*std(1-q*t*x)/(std(1-q*x)*std(1-t*x))
# weight for a fixed point in FHilb(line)
def weight(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res/(1-q)^(size)
    return factor(res)
# weight for a fixed point in FHilb(point)
def weightpt(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res/prod([(1-q*t*z[i-1]/z[i]) for i in range(1,size)])
    return res
# weight for a fixed point in Z(line)
def weightA(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res*prod([(1+a/z[i]) for i in range(size)])
    res=res/(1-q)^(size)
    return factor(res)
# weight for a fixed point in Z(pt)
def weightApt(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res*prod([(1+a/z[i]) for i in range(1,size)])
    res=res/prod([(1-q*t*z[i-1]/z[i]) for i in range(1,size)])
    return factor(res)
# weight for a line bundle
def bdle(z,po):
    size=len(z)
    res=prod([z[i]^(po[i]) for i in range(size)])
    return res
# Euler characteristic
def chi(po,opt):
    size=len(po)
    lst=StandardTableaux(size).list()
    if opt=="line":
        return sum([weight(zz(tab))*bdle(zz(tab),po) for tab in lst])
    if opt=="point":
        return sum([weightpt(zz(tab))*bdle(zz(tab),po) for tab in lst])
    if opt=="full":
        return sum([weightA(zz(tab))*bdle(zz(tab),po) for tab in lst])
    if opt=="fullpt":
        return sum([weightApt(zz(tab))*bdle(zz(tab),po) for tab in lst])
def chitest(po,j):
    size=len(po)
    lst=StandardTableaux(size).list()
    return sum([weight(zz(tab))*bdle(zz(tab),po)*(1-q)/(1-q*t*zz(tab)[j]/zz(tab)[j+1]) for tab in lst])
def chisimp(po):
    tmp=chi(po,"point")
    return tmp.subs({q:1,t:1})

HHH([0,0,0,0],[0,0,0,0,0,0])

(q^8*t*a + q^7*t^2*a + q^6*t^3*a + q^5*t^4*a + q^4*t^5*a + q^3*t^6*a + q^2*t^7*a + q*t^8*a + q^7*t*a^2 + q^6*t^2*a^2 + q^5*t^3*a^2 + q^4*t^4*a^2 + q^3*t^5*a^2 + q^2*t^6*a^2 + q*t^7*a^2 + q^8*t + q^7*t^2 + q^6*t^3 + q^5*t^4 + q^4*t^5 + q^3*t^6 + q^2*t^7 + q*t^8 - q^8*a + q^6*t^2*a + q^5*t^3*a + q^4*t^4*a + q^3*t^5*a + q^2*t^6*a - t^8*a - q^7*a^2 + q^5*t^2*a^2 + q^4*t^3*a^2 + q^3*t^4*a^2 + q^2*t^5*a^2 - t^7*a^2 + q^5*t*a^3 + q^4*t^2*a^3 + q^3*t^3*a^3 + q^2*t^4*a^3 + q*t^5*a^3 - q^8 - q^7*t - q*t^7 - t^8 - q^7*a - q^6*t*a + q^5*t^2*a + 2*q^4*t^3*a + 2*q^3*t^4*a + q^2*t^5*a - q*t^6*a - t^7*a - q^6*a^2 + 2*q^4*t^2*a^2 + 3*q^3*t^3*a^2 + 2*q^2*t^4*a^2 - t^6*a^2 - q^5*a^3 + q^3*t^2*a^3 + q^2*t^3*a^3 - t^5*a^3 - q^6*t + q^4*t^3 + q^3*t^4 - q*t^6 - q^6*a - 2*q^5*t*a - q^4*t^2*a - q^2*t^4*a - 2*q*t^5*a - t^6*a - 2*q^5*a^2 - 3*q^4*t*a^2 - 2*q^3*t^2*a^2 - 2*q^2*t^3*a^2 - 3*q*t^4*a^2 - 2*t^5*a^2 - q^4*a^3 - q^3*t*a^3 - q*t^3*a^3 - t^4*a^3 + q^2*t*a^4 + q*t^2*a^4 - q^5*t - 2*q^4*t^2 - 2*q^3*t^3 - 2*q